In [1]:
# @title 1. Environment Setup & Imports
# Installs necessary geospatial libraries and imports dependencies.
import os
import sys
import shutil
import glob
import json
import random
import re
import time
import tarfile
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

# Mount Google Drive
from google.colab import drive, userdata
if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("Drive already mounted.")

# Geospatial library for .tif handling
!pip install rasterio -q
import rasterio

# Set plotting style
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

Mounting Google Drive...
Mounted at /content/drive
PyTorch Version: 2.9.0+cu126
CUDA Available: True


In [2]:
# @title 2. Configuration (The Control Center)
# Define all paths, hyperparameters, and toggles here.

@dataclass(frozen=True)
class Config:
    # --- Paths ---
    drive_repo_root: str = "/content/drive/MyDrive/Colab_Notebooks/Training/baseline_models/UNet"
    # Google Drive Paths (Where your tar.gz files are)
    # 1. DATA PATH: Where your .tar.gz archives are stored
    drive_data_root: str = "/content/drive/MyDrive/Colab_Notebooks/Datasets/BioMassters/packed_compressed_data/tar_gz_archives"
    # Local Colab Paths (Where we extract data for speed)
    local_root: str = "/content/data"

    # Archive Filenames (Assumed names based on prompt)
    train_input_archive: str = "S1_July_train_features.tar.gz"
    train_label_archive: str = "labels/train_agbm.tar.gz"
    test_input_archive: str  = "S1_July_test_features.tar.gz"
    test_label_archive: str  = "labels/test_agbm.tar.gz"

        # --- Git Configuration ---
    repo_url: str = "github.com/MagicMorgigi/MA_baseline_UNet"
    user_email: str = "fmorgalla@yahoo.de"
    user_name: str = "MagicMorgigi"
    branch: str = "main"

    # --- Hyperparameters ---
    seed: int = 42
    batch_size: int = 16
    learning_rate: float = 1e-4
    num_epochs: int = 50
    patience: int = 10  # Early stopping

    # --- Data Processing ---
    standardize: bool = True
    stats_file: str = "standardization_stats.json" # Saved in output_dir

    # --- Loss Function Constraints ---
    # 1. High Pixel Value Exclusion
    exclude_high_values: bool = True
    exclusion_threshold: float = 400.0

    # 2. Weighted Range Penalty
    use_weighted_penalty: bool = False # Default disabled per prompt
    penalty_range: Tuple[float, float] = (5.0, 300.0)
    penalty_factor: float = 1.5

    exp_name = f"UNet_base_EP{num_epochs}_BS{batch_size}_LR{str(learning_rate).split('.')[1]}_SEED{seed}"

    if exclude_high_values:
        exp_name += f"_HiThresh{str(exclusion_threshold).split('.')[0]}"
    if use_weighted_penalty:
        exp_name += f"PenRange{str(penalty_range[0]).split('.')[0]}-{str(penalty_range[1]).split('.')[0]}\
        _PenFact{(int(str(penalty_factor).split('.')[0])-1)*100+(int(str(penalty_factor).split('.')[1])*10)}%"
    if not standardize:
        exp_name += "_NoStand"

    print(f"experiment name: {exp_name}")

    # Output Paths
    output_dir: str = f"/content/drive/MyDrive/Biomass_Project/experiments/{exp_name}"

cfg = Config()

# Create output directory if it doesn't exist
os.makedirs(cfg.output_dir, exist_ok=True)

# Seeding Function for Reproducibility
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False # Might slow down slightly but ensures repro

seed_everything(cfg.seed)
print("Configuration loaded and seeded.")

experiment name: UNet_base_EP50_BS16_LR0001_SEED42_HiThresh400
Configuration loaded and seeded.


In [3]:
# @title 3. Git Synchronization (Drive-Based)
# Syncs the specific Repository folder on Drive with GitHub.

def setup_git_drive(cfg: Config):
    try:
        pat = userdata.get('MA_colab_github_PAT')
    except:
        print("Warning: MA_colab_github_PAT not found in Colab Secrets. Git sync skipped.")
        return

    repo_path = cfg.drive_repo_root

    if not os.path.exists(repo_path):
        print(f"Error: Repo path {repo_path} does not exist. Please check Config.")
        return

    print(f"Configuring Git in: {repo_path}")

    # Configure Git Global
    !git config --global user.email "{cfg.user_email}"
    !git config --global user.name "{cfg.user_name}"

    # Navigate to Repo Folder
    os.chdir(repo_path)

    # Check if .git exists
    if not os.path.exists(os.path.join(repo_path, ".git")):
        print("Initializing new Git repository...")
        !git init
        !git branch -M {cfg.branch}
        auth_url = f"https://{pat}@{cfg.repo_url}"
        !git remote add origin {auth_url}

        try:
            !git pull origin {cfg.branch}
        except:
            print("Remote appears empty. Ready for first push.")
    else:
        print("Git repository detected. Updating credentials...")
        auth_url = f"https://{pat}@{cfg.repo_url}"
        !git remote set-url origin {auth_url}
        !git pull origin {cfg.branch}

    # .gitignore: Crucial to ignore the large model files in the experiment folder
    # if you don't want to push them to GitHub.
    gitignore_content = [
        "__pycache__/",
        "*.pyc",
        ".ipynb_checkpoints",
        "*.tif",            # Don't push image files
        "*.tar.gz",
        "experiments/",     # Ignore experiment outputs (models/plots)
        ".DS_Store"
    ]

    with open(".gitignore", "w") as f:
        f.write("\n".join(gitignore_content))

    print("Git setup complete.")
    os.chdir("/content") # Return to VM root for safety

setup_git_drive(cfg)

Configuring Git in: /content/drive/MyDrive/Colab_Notebooks/Training/baseline_models/UNet
Git repository detected. Updating credentials...
From https://github.com/MagicMorgigi/MA_baseline_UNet
 * branch            main       -> FETCH_HEAD
Already up to date.
Git setup complete.


In [4]:
# @title 4. Fast Data Extraction
# Copies archives from Data Drive folder to Local VM and extracts them.

def prepare_data(cfg: Config):
    # Note: source path now uses drive_data_root
    targets = {
        "train_input": (os.path.join(cfg.drive_data_root, cfg.train_input_archive), os.path.join(cfg.local_root, "train/inputs")),
        "train_label": (os.path.join(cfg.drive_data_root, cfg.train_label_archive), os.path.join(cfg.local_root, "train/labels")),
        "test_input":  (os.path.join(cfg.drive_data_root, cfg.test_input_archive),  os.path.join(cfg.local_root, "test/inputs")),
        "test_label":  (os.path.join(cfg.drive_data_root, cfg.test_label_archive),  os.path.join(cfg.local_root, "test/labels"))
    }

    if os.path.exists(cfg.local_root):
        if len(glob.glob(os.path.join(cfg.local_root, "train/inputs", "*.tif"))) > 0:
            print("Data appears to be already extracted. Skipping.")
            return

    os.makedirs(cfg.local_root, exist_ok=True)

    for key, (src_path, dest_dir) in targets.items():
        if not os.path.exists(src_path):
            print(f"CRITICAL: Archive not found at {src_path}")
            continue

        print(f"Processing {key}...")
        local_tar = os.path.join("/content", os.path.basename(src_path))

        # Copy
        if not os.path.exists(local_tar):
            print(f"  Copying from Drive Data folder...")
            shutil.copy(src_path, local_tar)

        # Extract
        os.makedirs(dest_dir, exist_ok=True)
        print(f"  Extracting...")
        try:
            with tarfile.open(local_tar, "r:gz") as tar:
                tar.extractall(path=dest_dir)
        except Exception as e:
            print(f"  Error extracting: {e}")

    print("Data preparation complete.")

prepare_data(cfg)

Processing train_input...
  Copying from Drive Data folder...
  Extracting...


/tmp/ipython-input-3217444835.py:38: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=dest_dir)


Processing train_label...
  Copying from Drive Data folder...
  Extracting...
Processing test_input...
  Copying from Drive Data folder...
  Extracting...
Processing test_label...
  Copying from Drive Data folder...
  Extracting...
Data preparation complete.


In [6]:
# @title 5. Dataset Class & Standardization Logic
# Handles regex mapping, TIF loading, and on-the-fly standardization.

class BiomassDataset(Dataset):
    def __init__(self, input_dir: str, label_dir: str, stats: Optional[Dict] = None):
        self.input_dir = input_dir
        self.label_dir = label_dir
        self.stats = stats

        # Get all input files
        self.input_files = sorted(glob.glob(os.path.join(input_dir, "*_S1_10.tif")))

        # Map inputs to labels using the 8-char ID
        self.data_pairs = []
        for inp_path in self.input_files:
            filename = os.path.basename(inp_path)
            # Regex to extract 8-char ID (alphanumeric)
            match = re.search(r"([a-zA-Z0-9]{8})_S1_10\.tif", filename)
            if match:
                file_id = match.group(1)
                label_name = f"{file_id}_agbm.tif"
                label_path = os.path.join(label_dir, label_name)

                if os.path.exists(label_path):
                    self.data_pairs.append((inp_path, label_path, file_id))

        print(f"Found {len(self.data_pairs)} valid pairs in {input_dir}")

    def __len__(self):
        return len(self.data_pairs)

    def __getitem__(self, idx):
        inp_path, lbl_path, file_id = self.data_pairs[idx]

        # Load Input (Read only first 2 bands: VV asc, VH asc)
        with rasterio.open(inp_path) as src:
            # Shape: (2, 256, 256)
            image = src.read([1, 2])
            # Replace NaNs if any (SAR usually safe, but good practice)
            image = np.nan_to_num(image, nan=0.0)

        # Load Label (1 band)
        with rasterio.open(lbl_path) as src:
            # Shape: (1, 256, 256)
            label = src.read(1)
            label = np.nan_to_num(label, nan=0.0)
            label = np.expand_dims(label, axis=0) # Ensure channel dim

        # Convert to Tensor
        image_t = torch.from_numpy(image).float()
        label_t = torch.from_numpy(label).float()

        # Standardization (On-the-fly)
        if self.stats:
            mean = torch.tensor(self.stats['mean']).view(2, 1, 1)
            std = torch.tensor(self.stats['std']).view(2, 1, 1)
            image_t = (image_t - mean) / (std + 1e-6)

        return image_t, label_t, file_id

def calculate_and_save_stats(dataset: Dataset, save_path: str):
    """Iterates through dataset to calculate band-wise mean/std."""
    print("Calculating statistics for standardization...")
    loader = DataLoader(dataset, batch_size=64, num_workers=2, shuffle=False)

    # Online algorithm for mean/std (Welford's or simple accumulation)
    # Simple accumulation is fine for this dataset size
    n_pixels = 0
    sum_ = torch.zeros(2)
    sq_sum = torch.zeros(2)

    for images, _, _ in tqdm(loader, desc="Scanning Dataset"):
        # images: [B, 2, H, W]
        b, c, h, w = images.shape
        num_pix = b * h * w

        # Calculate sum over batch, height, width
        sum_ += images.sum(dim=[0, 2, 3])
        sq_sum += (images ** 2).sum(dim=[0, 2, 3])
        n_pixels += num_pix

    mean = sum_ / n_pixels
    std = (sq_sum / n_pixels - mean ** 2).sqrt()

    stats = {
        "mean": mean.tolist(),
        "std": std.tolist()
    }

    with open(save_path, 'w') as f:
        json.dump(stats, f)

    print(f"Stats saved: {stats}")
    return stats

In [7]:
# @title 6. Model Architecture (Vanilla U-Net)
# Implements U-Net with learnable upsampling (ConvTranspose2d).

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels=2, n_classes=1):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes

        # Encoder
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))

        # Decoder (Learnable Upsampling)
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(1024, 512)

        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(256, 128)

        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up4 = DoubleConv(128, 64)

        # Output Head
        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        # Decoder
        x = self.up1(x5)
        # Skip connection concat (x4 + x)
        x = torch.cat([x4, x], dim=1)
        x = self.conv_up1(x)

        x = self.up2(x)
        x = torch.cat([x3, x], dim=1)
        x = self.conv_up2(x)

        x = self.up3(x)
        x = torch.cat([x2, x], dim=1)
        x = self.conv_up3(x)

        x = self.up4(x)
        x = torch.cat([x1, x], dim=1)
        x = self.conv_up4(x)

        logits = self.outc(x)
        return logits

In [8]:
# @title 7. Custom Loss Function
# Implements thresholding and weighted penalties.

class BiomassLoss(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.mse = nn.MSELoss(reduction='none')

    def forward(self, pred, target):
        # Basic pixel-wise MSE
        loss_map = self.mse(pred, target)

        # Create validity mask (exclude high values)
        valid_mask = torch.ones_like(target, dtype=torch.bool)
        if self.cfg.exclude_high_values:
            valid_mask = target <= self.cfg.exclusion_threshold

        # Apply Weighted Penalty
        if self.cfg.use_weighted_penalty:
            weights = torch.ones_like(target)
            low, high = self.cfg.penalty_range

            # Mask for range [5, 300]
            range_mask = (target >= low) & (target <= high)
            weights[range_mask] = self.cfg.penalty_factor

            loss_map = loss_map * weights

        # Apply Threshold Mask (Zero out loss for invalid pixels)
        loss_map = loss_map * valid_mask.float()

        # Calculate mean only over valid pixels
        num_valid = valid_mask.sum()
        if num_valid > 0:
            return loss_map.sum() / num_valid
        else:
            return torch.tensor(0.0, device=pred.device, requires_grad=True)

In [ ]:
# @title 8. Training Pipeline
# The main training loop with history tracking and plotting.

def train_pipeline(cfg: Config):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running on device: {device}")

    # --- 1. Dataset Setup ---
    train_input_path = os.path.join(cfg.local_root, "train/inputs")
    train_label_path = os.path.join(cfg.local_root, "train/labels")

    # Initialize raw dataset to calc stats
    full_train_dataset = BiomassDataset(train_input_path, train_label_path, stats=None)

    # Stats Loading / Calculation
    stats_path = os.path.join(cfg.output_dir, cfg.stats_file)
    if cfg.standardize:
        if os.path.exists(stats_path):
            print(f"Loading stats from {stats_path}")
            with open(stats_path, 'r') as f:
                stats = json.load(f)
        else:
            stats = calculate_and_save_stats(full_train_dataset, stats_path)
        full_train_dataset.stats = stats # Update stats

    # Split 80/20
    train_size = int(0.8 * len(full_train_dataset))
    val_size = len(full_train_dataset) - train_size
    train_subset, val_subset = random_split(full_train_dataset, [train_size, val_size])

    train_loader = DataLoader(train_subset, batch_size=cfg.batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_subset, batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)

    # --- 2. Model & Optimization ---
    model = UNet(n_channels=2, n_classes=1).to(device)
    criterion = BiomassLoss(cfg)
    optimizer = optim.Adam(model.parameters(), lr=cfg.learning_rate)

    # --- 3. Training Loop ---
    history = {
        'epoch': [], 'train_loss': [], 'val_loss': [],
        'val_rmse': [], 'val_mae': [], 'val_r2': []
    }

    best_val_loss = float('inf')
    epochs_no_improve = 0

    print("Starting training...")
    for epoch in range(cfg.num_epochs):
        model.train()
        train_loss_acc = 0.0

        # Training Step
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.num_epochs} [Train]", leave=False)
        for imgs, lbls, _ in pbar:
            imgs, lbls = imgs.to(device), lbls.to(device)

            optimizer.zero_grad()
            preds = model(imgs)
            loss = criterion(preds, lbls)
            loss.backward()
            optimizer.step()

            train_loss_acc += loss.item()
            pbar.set_postfix({'loss': loss.item()})

        avg_train_loss = train_loss_acc / len(train_loader)

        # Validation Step
        model.eval()
        val_loss_acc = 0.0

        # Metric accumulators
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for imgs, lbls, _ in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                preds = model(imgs)
                loss = criterion(preds, lbls)
                val_loss_acc += loss.item()

                # Apply high value exclusion for metrics calculation to be fair
                if cfg.exclude_high_values:
                    mask = lbls <= cfg.exclusion_threshold
                    all_preds.append(preds[mask].cpu())
                    all_targets.append(lbls[mask].cpu())
                else:
                    all_preds.append(preds.flatten().cpu())
                    all_targets.append(lbls.flatten().cpu())

        avg_val_loss = val_loss_acc / len(val_loader)

        # Calculate Metrics globally for epoch
        y_pred = torch.cat(all_preds).numpy()
        y_true = torch.cat(all_targets).numpy()

        mse = np.mean((y_true - y_pred)**2)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(y_true - y_pred))

        # R2 Calculation
        ss_res = np.sum((y_true - y_pred)**2)
        ss_tot = np.sum((y_true - np.mean(y_true))**2)
        r2 = 1 - (ss_res / (ss_tot + 1e-8)) # Avoid div by zero

        # Update History
        history['epoch'].append(epoch + 1)
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_rmse'].append(rmse)
        history['val_mae'].append(mae)
        history['val_r2'].append(r2)

        print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}")

        # Checkpointing & Early Stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), os.path.join(cfg.output_dir, "best_model.pth"))
            # Save metrics of best model
            with open(os.path.join(cfg.output_dir, "best_metrics.txt"), "w") as f:
                f.write(f"Epoch: {epoch+1}\nRMSE: {rmse}\nMAE: {mae}\nR2: {r2}\nVal Loss: {avg_val_loss}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= cfg.patience:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break

    # --- 4. Save History & Plots ---
    df_hist = pd.DataFrame(history)
    df_hist.to_csv(os.path.join(cfg.output_dir, "training_history.csv"), index=False)

    # Plotting
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    # Loss
    axes[0,0].plot(df_hist['epoch'], df_hist['train_loss'], label='Train MSE')
    axes[0,0].plot(df_hist['epoch'], df_hist['val_loss'], label='Val MSE')
    axes[0,0].set_title('Loss (MSE)')
    axes[0,0].legend()
    # RMSE
    axes[0,1].plot(df_hist['epoch'], df_hist['val_rmse'], color='orange')
    axes[0,1].set_title('Validation RMSE')
    # MAE
    axes[1,0].plot(df_hist['epoch'], df_hist['val_mae'], color='green')
    axes[1,0].set_title('Validation MAE')
    # R2
    axes[1,1].plot(df_hist['epoch'], df_hist['val_r2'], color='purple')
    axes[1,1].set_title('Validation R2')

    plt.tight_layout()
    plt.savefig(os.path.join(cfg.output_dir, "training_plots.png"))
    plt.show()

    return model

# --- EXECUTE TRAINING ---
# Comment out if only running inference after a restart
trained_model = train_pipeline(cfg)

Running on device: cpu
Found 8689 valid pairs in /content/data/train/inputs
Calculating statistics for standardization...


Scanning Dataset:   0%|          | 0/136 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


In [ ]:
# @title 9. Inference & Visualization Pipeline
# Runs evaluation on the Test set and visualizes predictions vs targets.

def inference_pipeline(cfg: Config, num_visualize=3):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 1. Load Data
    test_input_path = os.path.join(cfg.local_root, "test/inputs")
    test_label_path = os.path.join(cfg.local_root, "test/labels")

    # Load Stats
    stats_path = os.path.join(cfg.output_dir, cfg.stats_file)
    with open(stats_path, 'r') as f:
        stats = json.load(f)

    test_dataset = BiomassDataset(test_input_path, test_label_path, stats=stats)
    test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=2)

    # 2. Load Model
    model = UNet(n_channels=2, n_classes=1).to(device)
    model_path = os.path.join(cfg.output_dir, "best_model.pth")
    if not os.path.exists(model_path):
        print("No best model found. Run training first.")
        return

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # 3. Calculate Metrics
    all_preds = []
    all_targets = []

    print("Evaluating on Test Set...")
    with torch.no_grad():
        for imgs, lbls, _ in tqdm(test_loader):
            imgs = imgs.to(device)
            preds = model(imgs)

            # Apply same exclusion for fair metric comparison
            if cfg.exclude_high_values:
                mask = lbls <= cfg.exclusion_threshold
                all_preds.append(preds.cpu()[mask])
                all_targets.append(lbls[mask])
            else:
                all_preds.append(preds.cpu().flatten())
                all_targets.append(lbls.flatten())

    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_targets).numpy()

    mse = np.mean((y_true - y_pred)**2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))

    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    r2 = 1 - (ss_res / (ss_tot + 1e-8))

    print(f"TEST METRICS :: RMSE: {rmse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")

    # Append to metrics file
    with open(os.path.join(cfg.output_dir, "test_metrics.txt"), "w") as f:
         f.write(f"TEST RESULTS\nRMSE: {rmse}\nMAE: {mae}\nR2: {r2}")

    # 4. Visualization
    print(f"\nVisualizing {num_visualize} Examples...")

    # Pick random indices
    indices = np.random.choice(len(test_dataset), num_visualize, replace=False)

    fig, axes = plt.subplots(num_visualize, 2, figsize=(10, 5 * num_visualize))
    if num_visualize == 1: axes = [axes] # Handle singular case

    for i, idx in enumerate(indices):
        img, lbl, file_id = test_dataset[idx]

        # Predict
        input_tensor = img.unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(input_tensor).squeeze().cpu().numpy()

        target = lbl.squeeze().numpy()

        # Plot Target
        ax_tgt = axes[i][0]
        im1 = ax_tgt.imshow(target, cmap='viridis', vmin=0, vmax=400) # Fixed scale for consistency
        ax_tgt.set_title(f"ID: {file_id}\nTarget AGB")
        ax_tgt.axis('off')
        plt.colorbar(im1, ax=ax_tgt, fraction=0.046, pad=0.04)

        # Plot Prediction
        ax_pred = axes[i][1]
        im2 = ax_pred.imshow(pred, cmap='viridis', vmin=0, vmax=400)
        ax_pred.set_title(f"Predicted AGB")
        ax_pred.axis('off')
        plt.colorbar(im2, ax=ax_pred, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(os.path.join(cfg.output_dir, "test_visualizations.png"))
    plt.show()

inference_pipeline(cfg, num_visualize=5)

In [ ]:
# @title 10. Git Push Helper (Drive-Based)
def git_push_results(cfg: Config, message="Update training results"):
    repo_path = cfg.drive_repo_root

    if not os.path.exists(repo_path):
        print("Repository path not found.")
        return

    print(f"Pushing from: {repo_path}")
    os.chdir(repo_path)
    !git add .
    !git commit -m "{message}"

    print("Pushing to remote...")
    try:
        !git push origin {cfg.branch}
        print("Push complete!")
    except Exception as e:
        print(f"Push failed: {e}")

    os.chdir("/content")